# Dialforge Cloud Benchmark v4

Fresh uncached launcher for the Dialforge local AI benchmark. It tests **Qwen 3 1.7B, 4B and 8B**, **faster-whisper small.en**, **Chatterbox Nano**, and the full **STT → LLM → TTS** pipeline for all three Qwen tiers.

1. Choose **Runtime → Change runtime type → T4 GPU**.
2. Click **Runtime → Run all**.
3. Leave the tab open until the HTML report appears.

**Launcher version: v4-live-log.** Every bootstrap line is streamed live and also written to `/content/dialforge-bootstrap.log`.

In [ ]:
# DIALFORGE BENCHMARK v4-live-log
import pathlib, subprocess, sys, time, urllib.request
from IPython.display import HTML, display

LAUNCHER_VERSION = 'v4-live-log'
BOOTSTRAP_BASE = 'https://raw.githubusercontent.com/SumamaAhmed69/Axemetric-Caller-Beta-Runtime/main/benchmarks/dialforge_colab_bootstrap.py'
BOOTSTRAP = pathlib.Path('/content/dialforge_colab_bootstrap.py')
LOG = pathlib.Path('/content/dialforge-bootstrap.log')
REPORT = pathlib.Path('/content/dialforge-benchmark/dialforge-benchmark-report.html')

print('Dialforge launcher:', LAUNCHER_VERSION, flush=True)
print('Downloading current bootstrap without using a cached notebook copy...', flush=True)
last_error = None
for attempt in range(1, 6):
    try:
        url = BOOTSTRAP_BASE + f'?v={time.time_ns()}'
        req = urllib.request.Request(url, headers={'User-Agent': 'Dialforge-Colab-v4', 'Cache-Control': 'no-cache'})
        with urllib.request.urlopen(req, timeout=60) as response:
            payload = response.read()
        if len(payload) < 5000 or not payload.startswith(b'#!/usr/bin/env python3'):
            raise RuntimeError(f'Unexpected bootstrap payload ({len(payload)} bytes)')
        compile(payload.decode('utf-8'), str(BOOTSTRAP), 'exec')
        BOOTSTRAP.write_bytes(payload)
        print(f'Bootstrap verified: {len(payload)} bytes', flush=True)
        break
    except Exception as exc:
        last_error = exc
        print(f'Bootstrap download attempt {attempt}/5 failed: {type(exc).__name__}: {exc}', flush=True)
        time.sleep(2 * attempt)
else:
    raise RuntimeError(f'Could not download a valid Dialforge bootstrap: {last_error}')

print('\nStarting Dialforge bootstrap with LIVE output...', flush=True)
tail = []
with LOG.open('w', encoding='utf-8') as log:
    process = subprocess.Popen(
        [sys.executable, '-u', str(BOOTSTRAP)],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='', flush=True)
        log.write(line)
        log.flush()
        tail.append(line)
        if len(tail) > 160:
            tail.pop(0)
    returncode = process.wait()

if returncode != 0:
    print('\n===== BOOTSTRAP FAILURE TAIL =====', flush=True)
    print(''.join(tail), flush=True)
    print('Full bootstrap log:', LOG, flush=True)
    raise RuntimeError(f'Dialforge bootstrap exited with code {returncode}. The exact child-process output is shown above.')

if not REPORT.exists():
    raise RuntimeError(f'Bootstrap returned success but report is missing: {REPORT}')
print('\n=== DIALFORGE BENCHMARK COMPLETE ===', flush=True)
display(HTML(REPORT.read_text(encoding='utf-8')))


### Output files
- `/content/dialforge-benchmark/dialforge-benchmark-report.html`
- `/content/dialforge-benchmark/dialforge-benchmark-report.json`
- `/content/dialforge-bootstrap.log`

If the benchmark fails, the notebook prints the last 160 bootstrap lines automatically, so a screenshot of the bottom of the cell is enough to diagnose it.